In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
map_data = pd.read_csv('/mnt/netapp1/posadalab/ryan/summary_combined_stats.tsv', delimiter='\t')
read_counts = pd.read_csv('sample_read_counts.tsv', delimiter='\t')

overview = map_data.groupby('Sample_ID')[['Mapped_Reads']].sum().reset_index()
overview = overview.merge(read_counts, left_on='Sample_ID', right_on='Sample', how="left")
overview['Mapping_Rate'] = overview['Mapped_Reads'] / overview['ReadCount']
overview = overview[['Sample_ID', 'Mapped_Reads', 'ReadCount', 'Mapping_Rate']]
overview = overview.rename(columns={'Sample_ID':'filename', 'Mapped_Reads':'viral_reads', 'ReadCount':'total_reads', 'Mapping_Rate':'reads_percent_viral'})

matrix = map_data[map_data['Reference_Sequence'] != '*']
matrix = matrix.pivot(index='Reference_Sequence', columns='Sample_ID', values='Mapped_Reads')
matrix = matrix.fillna(0)
matrix = matrix.astype(int)
matrix

Sample_ID,0168-1Q-virome_S7_L001,0168-1Z-virome_S6_L001,0168-U-virome_S13_L001,14-2820AC-VIROME_S5_L001,14-2820AC-viromeTruSeq,15-8466AC-VIROME_S6_L001,15-8466AC-viromeTruSeq,2820-2Q-virome_S8_L001,2820-2Z-virome_S2_L001,2820-U-virome_S14_L001,...,S3-9504-VIROME_S3_L001,S3-9504-viromeTruSeq,S4-1876-VIROME_S4_L001,S4-1876-viromeTruSeq,SAC1-viromeR_S55_L001,SAC2-viromeR_S56_L001,SAD1-viromeR_S61_L001,SAD2-virome,SBC1-virome,SBC2-virome
Reference_Sequence,,,,,,,,,,,,,,,,,,,,,
14-2820AC-VIROME_S5_L001_k141_125_14-2820AC-VIROME_S5_L001,1,11,4,360886,100049,44,1,2,0,107198,...,0,6,220,43,0,0,0,0,0,0
14-2820AC-VIROME_S5_L001_k141_131_14-2820AC-VIROME_S5_L001,1,9,1,75937,62975,0,0,12,0,54548,...,0,2,0,0,0,0,0,0,0,0
14-2820AC-VIROME_S5_L001_k141_133_14-2820AC-VIROME_S5_L001,3,100,19,1089214,200645,0,1,3,2,514580,...,0,6,0,2,0,0,0,0,0,0
15-8466AC-VIROME_S6_L001_k141_66_15-8466AC-VIROME_S6_L001,0,0,0,2,27,1178213,361977,0,0,1,...,0,4,968,558,0,0,0,0,0,0
15-8466AC-VIROME_S6_L001_k141_67_15-8466AC-VIROME_S6_L001,0,0,0,0,15,1208541,290196,0,0,0,...,0,2,0,9,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UHGV-2236928,0,0,0,7,15,0,0,0,0,5,...,0,0,1,0,0,0,0,0,0,0
UHGV-2239868,8,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
UHGV-2241122,3,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
overview

,filename,viral_reads,total_reads,reads_percent_viral
0,0168-1Q-virome_S7_L001,645200,1296291,0.497728
1,0168-1Z-virome_S6_L001,831976,1588890,0.523621
2,0168-U-virome_S13_L001,1227563,1326604,0.925342
3,14-2820AC-VIROME_S5_L001,3143958,3215899,0.977630
4,14-2820AC-viromeTruSeq,824375,1333118,0.618381
5,15-8466AC-VIROME_S6_L001,3434672,3540653,0.970067
6,15-8466AC-viromeTruSeq,1127492,1320934,0.853557
7,2820-2Q-virome_S8_L001,636006,1261752,0.504066
8,2820-2Z-virome_S2_L001,918230,1510189,0.608023
9,2820-U-virome_S14_L001,1371965,1445127,0.949373


In [4]:
overview.to_csv('UHGV_tables/overview.tsv', sep='\t')

In [5]:
uhgv_metadata = pd.read_csv('/mnt/lustre/hsm/nlsas/notape/home/uvi/be/posadalab/loretta/UHGV/votus_metadata.tsv', delimiter='\t')
print(len(uhgv_metadata['ictv_taxonomy'].unique()))
uhgv_metadata = uhgv_metadata[['uhgv_genome', 'ictv_taxonomy']].rename(columns={'uhgv_genome': 'genome_id', 'ictv_taxonomy':'ictv_lineage'})

uhgv_labels = matrix.index[matrix.index.str.startswith('UHGV')]
uhgv_metadata = uhgv_metadata[uhgv_metadata['genome_id'].isin(uhgv_labels)]

uhgv_metadata_study = pd.read_csv('/mnt/lustre/hsm/nlsas/notape/home/uvi/be/posadalab/loretta/UHGV/ncbi_plus_uhgv_plus_contigs/uhgv_annotations_study/taxon_info.tsv', delimiter='\t')
uhgv_metadata_study['ictv_lineage'] = uhgv_metadata_study['ictv_lineage'].str.split().str[0]
uhgv_metadata_study = uhgv_metadata_study[['genome_id', 'ictv_lineage']]
uhgv_metadata_study

351


,genome_id,ictv_lineage
0,14-2820AC-VIROME_S5_L001_k141_133_14-2820AC-VI...,r__Monodnaviria;k__Sangervirae;p__Phixviricota...
1,14-2820AC-VIROME_S5_L001_k141_125_14-2820AC-VI...,r__Monodnaviria;k__Sangervirae;p__Phixviricota...
2,14-2820AC-VIROME_S5_L001_k141_130_14-2820AC-VI...,r__Monodnaviria;k__Sangervirae;p__Phixviricota...
3,14-2820AC-VIROME_S5_L001_k141_131_14-2820AC-VI...,r__Monodnaviria;k__Sangervirae;p__Phixviricota...
4,14-2820AC-viromeTruSeq_k141_767_14-2820AC-viro...,r__Monodnaviria;k__Sangervirae;p__Phixviricota...
...,...,...
103,SAD2-virome_k141_26_SAD2-virome,r__Monodnaviria;k__Sangervirae;p__Phixviricota...
104,SAD2-virome_k141_20_SAD2-virome,r__Monodnaviria;k__Sangervirae;p__Phixviricota...
105,SAD2-virome_k141_28_SAD2-virome,r__Monodnaviria;k__Sangervirae;p__Phixviricota...
106,SAD2-virome_k141_25_SAD2-virome,r__Monodnaviria;k__Sangervirae;p__Phixviricota...


In [6]:
suffix_map = {
    '-viria': 'r__',
    '-virae': 'k__',
    '-viricota': 'p__',
    '-viricetes': 'c__',
    '-virales': 'o__',
    '-viridae': 'f__',
    '-virus': 'g__',
}

def convert_lineage(lineage):
    parts = lineage.split(';')
    new_parts = []
    for p in parts[1:]:  # skip 'root'
        assigned = False
        for suffix, prefix in suffix_map.items():
            if p.endswith(suffix.replace('-', '')):
                new_parts.append(f"{prefix}{p}")
                assigned = True
                break
        if not assigned:
            # Could be species or unknown
            new_parts.append(f"s__{p}")
    return ';'.join(new_parts)

uhgv_metadata['ictv_lineage'] = uhgv_metadata['ictv_lineage'].apply(convert_lineage)
uhgv_metadata

,genome_id,ictv_lineage
0,UHGV-0121692,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...
1,UHGV-0169522,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...
2,UHGV-1375000,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...
3,UHGV-1368950,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...
4,UHGV-0062920,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...
...,...,...
160473,UHGV-0034819,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...
161030,UHGV-0081657,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...
161299,UHGV-1363755,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...
162190,UHGV-0023727,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...


In [7]:
uhgv_manual_annotations = pd.read_csv('/mnt/netapp1/posadalab/ryan/uhgv_virus_types.csv').rename(columns={'Unnamed: 0': 'ictv_lineage'})
uhgv_manual_annotations['ictv_lineage'] = uhgv_manual_annotations['ictv_lineage'].apply(convert_lineage)
uhgv_manual_annotations

,ictv_lineage,Genome type,Linear vs circular genome,Enveloped/non-enveloped,Genome size (kb),Size long or diameter (nm),Phage,Comment
0,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,dsDNA,NaN,non-enveloped,89.7-98.1,80,phage,Family not in ICTV or NCBI. Supposedly Alpha- ...
1,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,dsDNA,linear,non-enveloped,11.6-660,10-350,phage,Family not in ICTV or NCBI. Supposedly name de...
2,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,dsDNA,NaN,NaN,92.6-104.8,NaN,phage,Family not in ICTV or NCBI. Supposedly Delta- ...
3,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,dsDNA,linear,non-enveloped,11.6-660,10-350,phage,General information from AI
4,r__Monodnaviria;k__Sangervirae;p__Phixviricota...,ssDNA(+),circular,non-enveloped,4.4-6.1,30,phage,NaN
...,...,...,...,...,...,...,...,...
345,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,dsDNA,linear,non-enveloped,11.6-660,10-350,phage,General information of Caudoviricetes
346,r__Riboviria;k__Orthornavirae;p__Kitrinovirico...,ssRNA(+),linear,non-enveloped,4.2-4.6,25-33,no,NaN
347,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,dsDNA,linear,non-enveloped,11.6-660,10-350,phage,General information of Caudoviricetes
348,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,dsDNA,linear,non-enveloped,11.6-660,10-350,phage,General information of Caudoviricetes


In [8]:
full_metadata = pd.concat([uhgv_metadata, uhgv_metadata_study], ignore_index=True)

ranks = ['realm','kingdom','phylum','class','order','family','genus']

prefix_map = {
    'r__': 'realm',
    'k__': 'kingdom',
    'p__': 'phylum',
    'c__': 'class',
    'o__': 'order',
    'f__': 'family',
    'g__': 'genus'
}

def parse_lineage(lineage):
    out = {r: pd.NA for r in ranks}
    if pd.isna(lineage):
        return pd.Series(out)

    for part in lineage.split(';'):
        for pref, rank in prefix_map.items():
            if part.startswith(pref):
                val = part[len(pref):]
                out[rank] = val if val else pd.NA
    return pd.Series(out)

# parse correctly by prefix
split_df = full_metadata['ictv_lineage'].apply(parse_lineage)

# nearest assigned higher rank (to the left)
nearest_up = split_df.ffill(axis=1)

# build "Unclassified <nearest>" labels
for i, r in enumerate(ranks):
    if i == 0:
        split_df[r] = split_df[r].fillna("Unclassified root")
    else:
        split_df[r] = split_df[r].fillna("Unclassified " + nearest_up.iloc[:, i-1])

# merge back
full_metadata = pd.concat([full_metadata, split_df], axis=1)

In [9]:
full_metadata = full_metadata.merge(uhgv_manual_annotations, on='ictv_lineage', how='left')
full_metadata

,genome_id,ictv_lineage,realm,kingdom,phylum,class,order,family,genus,Genome type,Linear vs circular genome,Enveloped/non-enveloped,Genome size (kb),Size long or diameter (nm),Phage,Comment
0,UHGV-0121692,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,Duplodnaviria,Heunggongvirae,Uroviricota,Caudoviricetes,Crassvirales,Alpha/Gamma-crassviridae,Unclassified Alpha/Gamma-crassviridae,dsDNA,NaN,non-enveloped,89.7-98.1,80,phage,Family not in ICTV or NCBI. Supposedly Alpha- ...
1,UHGV-0169522,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,Duplodnaviria,Heunggongvirae,Uroviricota,Caudoviricetes,Unclassified Caudoviricetes,Flandersviridae,Unclassified Flandersviridae,dsDNA,linear,non-enveloped,11.6-660,10-350,phage,Family not in ICTV or NCBI. Supposedly name de...
2,UHGV-1375000,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,Duplodnaviria,Heunggongvirae,Uroviricota,Caudoviricetes,Unclassified Caudoviricetes,Flandersviridae,Unclassified Flandersviridae,dsDNA,linear,non-enveloped,11.6-660,10-350,phage,Family not in ICTV or NCBI. Supposedly name de...
3,UHGV-1368950,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,Duplodnaviria,Heunggongvirae,Uroviricota,Caudoviricetes,Unclassified Caudoviricetes,Flandersviridae,Unclassified Flandersviridae,dsDNA,linear,non-enveloped,11.6-660,10-350,phage,Family not in ICTV or NCBI. Supposedly name de...
4,UHGV-0062920,r__Duplodnaviria;k__Heunggongvirae;p__Uroviric...,Duplodnaviria,Heunggongvirae,Uroviricota,Caudoviricetes,Crassvirales,Alpha/Gamma-crassviridae,Unclassified Alpha/Gamma-crassviridae,dsDNA,NaN,non-enveloped,89.7-98.1,80,phage,Family not in ICTV or NCBI. Supposedly Alpha- ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57460,SAD2-virome_k141_26_SAD2-virome,r__Monodnaviria;k__Sangervirae;p__Phixviricota...,Monodnaviria,Sangervirae,Phixviricota,Malgrandaviricetes,Petitvirales,Microviridae,Unclassified Microviridae,ssDNA(+),circular,non-enveloped,4.4-6.1,30,phage,NaN
57461,SAD2-virome_k141_20_SAD2-virome,r__Monodnaviria;k__Sangervirae;p__Phixviricota...,Monodnaviria,Sangervirae,Phixviricota,Malgrandaviricetes,Petitvirales,Microviridae,Unclassified Microviridae,ssDNA(+),circular,non-enveloped,4.4-6.1,30,phage,NaN
57462,SAD2-virome_k141_28_SAD2-virome,r__Monodnaviria;k__Sangervirae;p__Phixviricota...,Monodnaviria,Sangervirae,Phixviricota,Malgrandaviricetes,Petitvirales,Microviridae,Unclassified Microviridae,ssDNA(+),circular,non-enveloped,4.4-6.1,30,phage,NaN
57463,SAD2-virome_k141_25_SAD2-virome,r__Monodnaviria;k__Sangervirae;p__Phixviricota...,Monodnaviria,Sangervirae,Phixviricota,Malgrandaviricetes,Petitvirales,Microviridae,Unclassified Microviridae,ssDNA(+),circular,non-enveloped,4.4-6.1,30,phage,NaN


In [10]:
full_metadata['Genome type'].value_counts()

Genome type
dsDNA         49332
ssDNA(+)       6609
ssDNA(-)        864
ssDNA(+/-)      601
ssRNA(+)         51
ssDNA             5
dsRNA             2
dsDNA-RT          1
Name: count, dtype: int64

In [11]:
reads_with_tax = matrix.merge(
    full_metadata.set_index('genome_id'),
    left_index=True, right_index=True,
    how='left'
)

# Define levels you want to aggregate
levels = ['class','order','family', 'Genome type']

# Dictionary to store aggregated matrices
agg_matrices = {}

for level in levels:
    # Group by the taxonomy level and sum counts
    agg = reads_with_tax.groupby(level)[matrix.columns].sum()
    agg_matrices[level] = agg

In [12]:
agg_matrices['Genome type']

,0168-1Q-virome_S7_L001,0168-1Z-virome_S6_L001,0168-U-virome_S13_L001,14-2820AC-VIROME_S5_L001,14-2820AC-viromeTruSeq,15-8466AC-VIROME_S6_L001,15-8466AC-viromeTruSeq,2820-2Q-virome_S8_L001,2820-2Z-virome_S2_L001,2820-U-virome_S14_L001,...,S3-9504-VIROME_S3_L001,S3-9504-viromeTruSeq,S4-1876-VIROME_S4_L001,S4-1876-viromeTruSeq,SAC1-viromeR_S55_L001,SAC2-viromeR_S56_L001,SAD1-viromeR_S61_L001,SAD2-virome,SBC1-virome,SBC2-virome
Genome type,,,,,,,,,,,,,,,,,,,,,
dsDNA,644322,831085,6087,7127,31064,57364,15207,634270,917242,8460,...,16582,79121,1163,21328,64,197,557,539,167,100
dsDNA-RT,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
dsRNA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ssDNA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ssDNA(+),692,760,1221456,3136797,792864,3377234,1111514,404,47,1363454,...,4324810,630971,4331397,1385685,318392,322425,157086,186403,240942,258890
ssDNA(+/-),158,56,19,26,431,74,758,1095,2,50,...,59,405,12,141,6733,6353,23203,26429,25431,20687
ssDNA(-),0,0,1,1,0,0,0,0,0,0,...,0,2,0,1,0,0,0,0,0,0
ssRNA(+),28,75,0,7,16,0,13,237,939,1,...,57,125,2,6,0,0,0,0,0,0


In [13]:
agg_matrices['Genome type'].to_csv('UHGV_tables/genome_type_data.tsv', sep='\t')

In [14]:
agg_matrices['family'].to_csv('UHGV_tables/family_data.tsv', sep='\t')
agg_matrices['order'].to_csv('UHGV_tables/order_data.tsv', sep='\t')
agg_matrices['class'].to_csv('UHGV_tables/class_data.tsv', sep='\t')

In [15]:
agg_matrices['Genome type'].sum(axis=1)

Genome type
dsDNA          8185208
dsDNA-RT             0
dsRNA                0
ssDNA                1
ssDNA(+)      36699044
ssDNA(+/-)     5642830
ssDNA(-)          1499
ssRNA(+)         12253
dtype: int64

In [19]:
full_metadata_no_microviridae_caudoviricetes = full_metadata[(full_metadata['family'] != 'Microviridae') & (full_metadata['class'] != 'Caudoviricetes')]

reads_with_tax = matrix.merge(
    full_metadata_no_microviridae_caudoviricetes.set_index('genome_id'),
    left_index=True, right_index=True,
    how='inner'
)

# Define levels you want to aggregate
levels = ['class','order','family', 'Genome type']

# Dictionary to store aggregated matrices
agg_matrices = {}

for level in levels:
    # Group by the taxonomy level and sum counts
    agg = reads_with_tax.groupby(level)[matrix.columns].sum()
    agg_matrices[level] = agg

In [24]:
agg_matrices['family'].to_csv('UHGV_tables/family_data_filtered.tsv', sep='\t')
agg_matrices['order'].to_csv('UHGV_tables/order_data_filtered.tsv', sep='\t')
agg_matrices['class'].to_csv('UHGV_tables/class_data_filtered.tsv', sep='\t')
agg_matrices['Genome type'].to_csv('UHGV_tables/genome_type_data_filtered.tsv', sep='\t')
reads_with_tax.to_csv('UHGV_tables/species_data_filtered.tsv', sep='\t')